## scATAC preprocessing

### Setup libraries

In [ ]:
import numpy as np
import snapatac2 as snap
import plotly.express as px
import pylab as pl
import sys
import terra_pandas as tpd
from collections import defaultdict
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
sys.path.append('./utils')
from importing import download_h5ads, add_metadata_to_adata, build_adata, add_gene_names_to_adata, run_scrublet
from plotting import plot_knee_curve, plot_expression_heatmap, stacked_barplot_proportions


In [ ]:
chrom_sizes_dict = defaultdict(int)

print('Loading chromosomes', file=sys.stderr)
with open("references/mm39.chrom.sizes", "r") as fh:
    for line in fh:
        if not ("random" in line.strip().split("\t")[0] or "Un" in line.strip().split("\t")[0]):
            chrom_sizes_dict[line.strip().split("\t")[0]] = int(line.strip().split("\t")[1])

chrom_sizes_dict.keys()

### Create snapATAC2 object and calculate basic QCs.

This assumes you have run the download ATAC fragments script. Here we create a snapATAC2 object with no filtering and plot the ATAC knee plot. 

Then we apply a very minimal filter of 10 fragments per barcode and calculate some basic QCs per barcode to plot. 

In [ ]:
ssecpkr_list = ["Subpool_1","Subpool_2","Subpool_3","Subpool_4","Subpool_5"]

fragment_files = [f"data/{ssecpkr}.fragments.tsv.gz" for ssecpkr in ssecpkr_list]
output_names = [f"data/{ssecpkr}_atac.h5ad" for ssecpkr in ssecpkr_list]
adatas = snap.pp.import_data(
    fragment_file=fragment_files,
    file=output_names,
    chrom_sizes=chrom_sizes_dict,
    sorted_by_barcode=False,
    min_num_fragments=0,
    shift_left=4,
    shift_right=-4
)

In [ ]:
adatas = []
output_names = [f"/home/jupyter/sharev2-adatas/{ssecpkr}_atac.h5ad" for ssecpkr in ssecpkr_list]
for ssecpkr in output_names:
    adatas.append(snap.read(ssecpkr))
print(adatas)

In [ ]:
import os 

fig, ax = plt.subplots(figsize=(8, 6))
fontsize = 18
colors = plt.cm.tab10.colors

for i,adata_subset in enumerate(adatas):
    umi_counts = np.array(adata_subset.obs["n_fragment"])
    umi_counts.sort()
    umi_counts = umi_counts[::-1]

    ax.loglog(range(len(umi_counts)), umi_counts, linewidth=2, label=os.path.basename(adata_subset.filename), color=colors[i % len(colors)])
    
ax.set_ylabel("UMI Counts", fontsize=10)
ax.set_xlabel("Barcodes", fontsize=10)
ax.tick_params(axis='both', which='major', labelsize=10)
ax.axhline(y=500, linewidth=2, color="#505050", linestyle='--', label='500 fragments')
ax.legend(fontsize=14)
plt.grid(True, which="both")


In [ ]:
snap.metrics.tsse(adatas, "references/genes.gtf")

In [ ]:
adataset.obs["n_fragment"] = adataset.adatas.obs['n_fragment']
adataset.obs["frac_dup"] = adataset.adatas.obs['frac_dup']
adataset.obs["frac_mito"] = adataset.adatas.obs['frac_mito']
adataset.obs["tsse"] = adataset.adatas.obs['tsse']

In [ ]:
ob = pd.DataFrame({ 'n_fragments': adataset.obs["n_fragment"], 
                   'tsse': adataset.obs["tsse"],
                  'frac_dup': adataset.obs["frac_dup"],
                  'frac_mito': adataset.obs["frac_mito"],
                  'sample':adataset.obs["sample"],
                  'barcode': adataset.obs_names})
ob.to_csv("/home/jupyter/unfilt-atac-obs.tsv", sep="\t", index=False)

In [ ]:
snap.pl.tsse(adataset, interactive=False)


In [ ]:
snap.pl.frag_size_distr(adataset, interactive=False)


### Now let's apply slightly more aggressive filtering, perform doublet assignment and do basic clustering.

After clustering, we can plot barplot showing Subpool proportions per leiden cluster. We can see which clusters are enriched in doublets, and validate by looking at total fragment counts per cluster.


In [ ]:
snap.pp.filter_cells(adatas, min_tsse=5, min_counts=500)
snap.pp.add_tile_matrix(adatas, bin_size=5000)
snap.pp.select_features(adatas, n_features=50000)
snap.pp.scrublet(adatas)

adataset.obs["doublet_probability"] = adataset.adatas.obs['doublet_probability']

In [ ]:
#plot doublet distribution

df_list = []
for adata,label in zip(adatas,output_names):
    df = pd.DataFrame({
        "doublet_probability": adata.obs['doublet_probability'],
        "sample": label
    })
    df_list.append(df)
combined_df = pd.concat(df_list, ignore_index=True)
sns.kdeplot(
    data=combined_df,
    x="doublet_probability",
    hue="sample",
    common_norm=False,
    fill=True,
    alpha=0.4
)
plt.title("Doublet Probability Distributions")
plt.xlabel("Doublet Probability")
plt.ylabel("Density")
plt.legend(title="Sample")
plt.tight_layout()


In [ ]:
snap.pp.select_features(adataset, n_features=50000)
snap.pp.knn(adataset)
snap.tl.leiden(adataset)

In [ ]:
snap.pl.umap(adataset, color="leiden", interactive=False, height=500, width=500)

In [ ]:
ob = pd.DataFrame({ 'n_fragments': adataset.obs["n_fragment"], 
                   'tsse': adataset.obs["tsse"],
                  'frac_dup': adataset.obs["frac_dup"],
                  'frac_mito': adataset.obs["frac_mito"],
                  'sample':adataset.obs["sample"],
                  'barcode': adataset.obs_names},
                 'doublet_probability': adataset.obs["doublet_probability"],
                'leiden' : adataset.obs["leiden"])
ob.loc[ob["doublet_probability"] >=0.7 ,"is_doublet"] = "True"
ob.to_csv("results/filt-atac-obs.tsv", sep="\t", index=False)

In [ ]:
#Subpool distribution by leiden
stacked_barplot_proportions(ob, cluster_key="leiden", var_key="sample", reverse_order=True)

In [ ]:
#Doublet assignment per cluster
stacked_barplot_proportions(ob, cluster_key="leiden", var_key="is_doublet", reverse_order=True)

In [ ]:
#violin plot of total fragments by leiden

sns.violinplot(data = ob, x = "leiden", y="n_fragment")
plt.yscale("log")

In [ ]:
atac_adata = adataset.to_adata()
atac_adata.write("results/filtered-ATAC.h5ad")